# Attention

### Masked Multi Head Attention

To understand the differnce lets review to Standard Multi-Head Attention (MHA).

#### Standard Multi-Head Attention (MHA)

Scaled Dot-Product Attention $softmax(\frac{QK^T}{√d}) V$

Now after `Scaled Dot-Product Attention` review of in both types, the core math is the same the `Scaled Dot-Product Attention` but the difference lies entirely in what tokens are allowed to "see" each other before the softmax step.

Here is the clearest breakdown of the differences and the specific use cases.

1. **Standard Multi-Head Attention (MHA)**

- Directionality: Bidirectional (Full visibility).

- The View: Every token in the input sequence can look at every other token—including the ones that come before it and the ones that come after it (past, present, and future).

- Information Flow: The attention scores matrix is fully populated.

- Where it's used: Primarily in the Encoder part of the Transformer (e.g., BERT). It is also used in the Cross-Attention sub-layer of the Decoder (where the decoder queries the encoder's output).

- Goal: Understanding. The model needs the entire surrounding context (left + right) to grasp the full meaning of a word (e.g., figuring out if "bank" means a river bank or a financial bank based on words on both sides).

2. **Masked Multi-Head Attention (MMHA)**

- Directionality: Unidirectional (Causal / Left-to-Right).

- The View: A token at position i can only look at positions <= i (itself and everything that came before it). It is strictly forbidden from looking at future tokens (> i).

- The Mask Mechanics: Before the softmax, we add a massive negative number (-inf) to the upper-right triangle of the attention score matrix. When softmax turns these into probabilities, those future positions become exactly 0% attention.

- Where it's used: In the Decoder's Self-Attention sub-layer (e.g., GPT, LLaMA).

- Goal: Generation. The model must predict the next word based only on the words it has already written.

When you train a model like GPT to write the next word, you use a technique called "Teacher Forcing." You feed the entire ground-truth sentence into the model at once to speed up training.

> Note: Teacher forcing used for trainig from scratch a model not only used for Knowledge Distillation

However, if the model could see the future words during training, it would simply "cheat" by copying the next token from the input, and it would never learn to actually predict.

The **Mask** solves this:

- When the model is predicting the 5th word in "The cat sat on the ___", the mask ensures it cannot see the 6th word (e.g., "mat") that we secretly fed into the input. It must generate the 5th word using only words 1 through 4.

- This forces the model to learn the actual patterns of language.

#### Implemetation

##### Core Scaled-Dot Product Attention

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q: Query (batch, heads, seq_len, d_k)
    k: Key (batch, heads, seq_len, d_k)
    v: Value (batch, heads, seq_len, d_k)
    mask: (seq_len, seq_len) or (batch, 1, 1, seq_len)
    """
    d_k = q.size(-1)
    
    # calculate raw scores
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores + mask

    # softmax
    attention_weights = F.softmax(scores, dim=-1)

    output = torch.matmul(attention_weights, v)

    return output, attention_weights

##### Multi-Head Attention Class

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model / num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def split_heads(self, x):
        batch_size, seq_len, _ = x.size()
        x = x.view(int(batch_size), int(seq_len), int(self.num_heads), int(self.d_k))
        return x.transpose(1, 2)
        
    def combine_heads(self, x):
        batch_size, _, seq_len, _ = x.size()
        x.transpose(1, 2)
        return x.contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_q(query))
        V = self.split_heads(self.W_v(value))
        K = self.split_heads(self.W_k(key))
        
        attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)

        output = self.W_o(self.combine_heads(attn_output))

        return output
        

##### Standard Multi-Head Attention (No Mask)

d_model = 512
num_heads = 8
batch_size = 4
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)

# Standard attention: every token sees every other token
output = mha(x, x, x, mask=None)  
print(output.shape)

##### Masked Multi-Head Attention (Causal Mask)

def create_causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
    return mask

causal_mask = create_causal_mask(seq_len)
print(causal_mask)
output = mha(x, x, x, mask=causal_mask)
output

### Cross Attention Mechanism

Cross-attention mechanism is a key part of the Transformer model. It allows the decoder to access and use relevant information from the encoder. This helps the model focus on important details, ensuring tasks like translation are accurate. `Imagine generating captions for images (decoder) from a detailed description (encoder). Cross-attention helps the caption generator focus on key details, ensuring accuracy in the caption.`

Let's use a simple analogy to explain the process:

1. **Encoder (English Story):** Encoder reads the English story and breaks it down into smaller chunks, like sentences or words. Each chunk is then turned into a "representation" that captures its meaning.
2. **Decoder (Spanish Translation):** Decoder's job is to create the Spanish translation, one word at a time. As it generates each word, it needs to know which part of the English story is most important for the current translation.
3. **Cross-Attention (Helper):** Cross-attention acts as a bridge between the encoder and decoder. It allows the decoder to "ask" the encoder which parts of the English story are most relevant for translating the current word, ensuring the translation is accurate and meaningful.

![image.png](attachment:250a3c56-c5c9-4dce-a4c5-9237ae776267.png)

Step-by-Step Process:

1. **Query (Question):** Decoder creates a "query" for each word it's trying to translate. This query is like a question asking, "Which part of the English story should I focus on?"
2. **Keys and Values (Answers):** Encoder provides "keys" and "values." The keys are like labels that help identify the important parts of the story and the values are the actual content of those parts.
3. **Matching (Finding the Best Fit):** Cross-attention mechanism compares the query from the decoder with the keys from the encoder. It calculates how well each query matches each key. This is like finding the best fit between the question and the answers.
4. **Combining Information (Putting It All Together):** Once the best matches are found, the cross-attention mechanism combines the relevant information from the encoder (values) to help the decoder generate the next word in the translation.

![attentions](./images/1.webp)

Cross attention is a key component in transformers, where a sequence can attend to another sequence’s information, making it essential for tasks like translation or summarization. In this blog, I’ll dive into how cross attention works, building upon concepts like self-attention, multi-head attention, and masked multi-head attention, which I’ve covered in previous blogs. Along with layer normalization and positional encoding, cross attention plays a crucial role in capturing relationships between sequences, enabling transformers to handle complex dependencies in tasks like machine translation.

Practical Applications of Cross Attention :

Cross-attention is particularly useful in scenarios where there are two distinct sequences involved. Some key applications include:

- Machine Translation: Translating text from one language to another involves comparing and aligning words between the source and target languages.
- Question Answering: Determining which parts of the context are relevant to answer a given question.
- Image Captioning: Generating descriptive text for images, where the image and text are treated as different modalities.
- Text-to-Image Generation: Creating images based on textual descriptions.
- Text-to-Speech: Generating speech from text, where the input is text and the output is speech.

![attentions](./images/2.webp)

### Papers:

#### Cross-Attention is All You Need: Adapting Pretrained Transformers for Machine Translation

https://arxiv.org/html/2104.08771v2

We study the power of cross-attention in the Transformer architecture within the context of transfer learning for machine translation, and extend the findings of studies into cross-attention when training from scratch. We conduct a series of experiments through fine-tuning a translation model on data where either the source or target language has changed. **These experiments reveal that fine-tuning only the cross-attention parameters is nearly as effective as fine-tuning all parameters** (i.e., the entire translation model). We provide insights into why this is the case and observe that limiting fine-tuning in this manner yields cross-lingually aligned embeddings. The implications of this finding for researchers and practitioners include a mitigation of catastrophic forgetting, the potential for zero-shot translation, and the ability to extend machine translation models to several new language pairs with reduced parameter storage overhead.

Our code is available at https://github.com/MGheini/xattn-transfer-for-mt.

#### Cross-Attention and Encoder–Decoder Transformers: A Logical Characterization

https://arxiv.org/html/2605.07705v1